# Time-Series Sales Forecasting (Holt-Winters & ARIMA)

Performs time series decomposition, stationarity testing, and exponential smoothing forecasts for multi-horizon demand planning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller


## 1. Load Daily Sales Time Series


In [ ]:
orders_path = Path('../data/raw/orders.csv')
df = pd.read_csv(orders_path)
df['order_date'] = pd.to_datetime(df['order_date'])

daily_sales = df[df['status'] != 'Cancelled'].groupby('order_date')['total_amount'].sum()
weekly_sales = daily_sales.resample('W').sum()

plt.figure(figsize=(14, 5))
plt.plot(weekly_sales.index, weekly_sales.values, color='#2563eb', linewidth=2)
plt.title('Weekly Enterprise Sales Trajectory', fontweight='bold')
plt.ylabel('Revenue ($)')
plt.show()


## 2. Augmented Dickey-Fuller (ADF) Stationarity Test


In [ ]:
adf_res = adfuller(weekly_sales.dropna())
print(f"ADF Statistic: {adf_res[0]:.4f}")
print(f"p-value: {adf_res[1]:.4f}")


## 3. Holt-Winters Exponential Smoothing Model


In [ ]:
hw_model = ExponentialSmoothing(
    weekly_sales,
    trend='add',
    damped_trend=True
).fit(damping_trend=0.95)

forecast_12w = hw_model.forecast(12)
plt.figure(figsize=(14, 6))
plt.plot(weekly_sales.index[-40:], weekly_sales.values[-40:], label='Historical Sales', color='#334155')
plt.plot(forecast_12w.index, forecast_12w.values, label='12-Week Projected Forecast', color='#2563eb', linestyle='--', linewidth=2.5)
plt.title('Holt-Winters 12-Week Sales Forecast', fontweight='bold')
plt.legend()
plt.show()
